# Day 3: One Whiteboard, Many Writers. Who Owns What?

**Today's goal:** give SupportDesk a *typed, shared state* where every value has an owner and a merge rule, then split the one big graph into two small, testable **subgraphs** without changing a single answer.

## The big analogy: a shared whiteboard in an office

Picture one whiteboard that several people write on while handling a customer's request.

| On the whiteboard | In our agent | Today's word |
| --- | --- | --- |
| The board itself, with labelled boxes | The agent's `State` | **State** |
| A rule: *"Only the retrieval desk writes in the CONTEXT box"* | `OWNS` + `enforce_contract` | **Ownership / contract** |
| What happens when two people write in the same box: **erase and replace**, **add below**, or **add only if new** | No reducer, `operator.add`, `merge_unique` | **Reducer** |
| A **department** with its own small whiteboard | A compiled graph with its own state class | **Subgraph** |
| A **template** for building a department from a list of tasks | `make_linear_graph` | **Factory** |
| The **receptionist** who decides which department sees the customer first | The parent graph's routing | **Orchestration** |
| A **photocopy** handed to the department, and only the result copied back | `call_subgraph` | **Wrapper** |

**The one idea to remember:** a shared value without an owner and a merge rule is a bug waiting to happen. Write the rule down, then enforce it in code.

## How to use this notebook
1. Run each cell from top to bottom: click a cell, then press **Shift + Enter**.
2. Read the short story above each cell first.
3. Nothing here calls an AI model. It runs offline in a few seconds. The "tool" is a local stand-in for the real MCP call you build on Days 6 to 8.
4. In **Step 8** you will *predict* before you run.


## Step 0: Setup

We import `operator` (for the built-in "add below" rule), `re` (to split a question into words), `Annotated` and `TypedDict` (to describe boxes and their merge rules), and `langgraph`.

In [1]:
import operator
import re
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

print("Setup done.")

Setup done.


## Step 1: The small library of facts

SupportDesk answers bank questions. Here is the tiny knowledge base it uses:

- `POLICY_WORDS`: words that tell us *"this is a policy question, look it up"*.
- `TOPICS`: a quick word-to-document table (a second way to find the right document).
- `DOCS`: four policy documents. **P2 is expired** (a retired rule), so retrieval must ignore it.

In [2]:
POLICY_WORDS = {"policy", "fee", "fees", "charge", "rule", "loan", "prepayment", "lost", "stolen", "report"}
TOPICS = {"lost": "P1", "stolen": "P1", "loan": "P3", "prepayment": "P3", "fee": "P4"}
DOCS = [
    {"id": "P1", "status": "current", "text": "Lost or stolen cards must be reported within 24 hours. A frozen card is unfrozen at a branch."},
    {"id": "P2", "status": "expired", "text": "Lost cards can be reported by phone only (rule retired in 2024)."},
    {"id": "P3", "status": "current", "text": "Personal loan prepayment charge is 2 percent after 12 months."},
    {"id": "P4", "status": "current", "text": "The monthly account fee is waived above 10,000 INR average balance."},
]
print(len(DOCS), "documents loaded;", sum(d["status"] == "current" for d in DOCS), "are current.")

4 documents loaded; 3 are current.


## Step 2: Merge rules (reducers) and the typed whiteboard

### What is a reducer?
When a step finishes, it returns a small update such as `{"context": ["P1"]}`. LangGraph must decide how to combine that with what is already in the box. That decision is the **reducer**.

| Rule | Whiteboard version | Code |
| --- | --- | --- |
| **Overwrite** (the default) | Erase the box and write the new thing | no reducer |
| **Append** | Add the new lines below the old ones | `operator.add` |
| **Append if new** | Add only the items that are not already there | `merge_unique` (we write it) |

Two boxes get a rule:
- `trace` (a log: every step adds a line) uses **append**.
- `context` (the documents found) uses **append if new**, because two different retrievers may find the *same* document, and we do not want it twice.

Everything else is overwritten by its one owner.

### The three whiteboards
- `State`: the big board for the whole agent (the same field names as the Day 19 capstone).
- `RetrievalState`: the retrieval department's small board.
- `ToolState`: the tool department's small board.

`TypedDict` only *describes* the boxes. Python does **not** check it while the program runs. Remember that for Step 3.

In [3]:
def merge_unique(old, new):
    """Custom reducer: keep the old items, add only new items we do not already have."""
    return old + [item for item in new if item not in old]


class State(TypedDict):          # the whole SupportDesk whiteboard
    user: str
    team: str
    question: str
    cid: str
    t0: float
    route: str
    context: Annotated[list, merge_unique]   # two retrievers write it: append if new
    tool: str
    result: str
    answer: str
    tokens: int
    blocked: str
    fallback: bool
    trace: Annotated[list, operator.add]     # many steps write it: append


class RetrievalState(TypedDict): # the retrieval department's own small board
    question: str
    cid: str
    context: Annotated[list, merge_unique]
    trace: Annotated[list, operator.add]


class ToolState(TypedDict):      # the tool department's own small board
    question: str
    cid: str
    tool: str
    result: str
    trace: Annotated[list, operator.add]

print("State has", len(State.__annotations__), "boxes; RetrievalState has", len(RetrievalState.__annotations__),
      "; ToolState has", len(ToolState.__annotations__), ".")

State has 14 boxes; RetrievalState has 4 ; ToolState has 5 .


## Step 3: Ownership, enforced (the contract)

**Analogy.** On the whiteboard there is a sticker on every box saying *who may write here*. Without a guard, a rushed colleague can still scribble in someone else's box.

Because `TypedDict` is only a hint, we add a small **guard** called `enforce_contract(owner, update)`. It checks two things:
1. **Is this owner allowed to write that key?** (`OWNS` is the list of allowed keys per owner.)
2. **Is the value the right type?** (`TYPES` says what each key must be.)

If not, it fails loudly with an error. A quiet wrong value is much more expensive than a loud error.

The table below is drawn from `OWNS`. Read it as: **"column X may write row Y"**.

In [4]:
OWNS = {"retrieval": {"context", "trace"},
        "tool_path": {"tool", "result", "trace"},
        "write_answer": {"answer", "trace"}}
TYPES = {"context": list, "trace": list, "tool": str, "result": str, "answer": str}


def enforce_contract(owner, update):
    for key, value in update.items():
        if key not in OWNS[owner]:
            raise PermissionError(owner + " may not write '" + key + "'")
        if not isinstance(value, TYPES[key]):
            raise TypeError(owner + " wrote '" + key + "' as " + type(value).__name__)
    return update


# A small picture of who may write what (X = allowed).
owners = list(OWNS)
print(f"{'box':10}" + "".join(f"{o:14}" for o in owners))
for key in ["context", "tool", "result", "answer", "trace"]:
    print(f"{key:10}" + "".join(f"{('X' if key in OWNS[o] else '.'):14}" for o in owners))

box       retrieval     tool_path     write_answer  
context   X             .             .             
tool      .             X             .             
result    .             X             .             
answer    .             .             X             
trace     X             X             X             


## Step 4: The workers (the nodes)

Six small functions. The important thing: **these exact functions are used twice**, once in the old single graph (the monolith) and once in the new split graph. If the answers stay the same, the refactor is safe.

| Worker | Job |
| --- | --- |
| `decide_route` | Is this a policy question (`rag`) or a tool question (`tool`)? |
| `retrieve` | Find the best **current** document by overlapping words |
| `retrieve_by_topic` | A second way: look the topic words up in a table (it may find the *same* document again) |
| `choose_tool` | `freeze_card`, `lookup_balance`, or `none` |
| `run_tool` | A stand-in for the real MCP call (built on Days 6 to 8) |
| `write_answer` | Combine context and result into the final answer |

Each worker also adds one line to `trace` in the form `cid=<id> <node> <detail>`. (Day 4 adds timing to these lines.)

In [5]:
def words_of(text):
    return set(re.findall(r"[a-z]+", text.lower()))


def decide_route(state):
    route = "rag" if words_of(state["question"]) & POLICY_WORDS else "tool"
    return {"route": route, "trace": ["cid=" + state["cid"] + " decide_route route=" + route]}


def retrieve(state):   # keyword overlap over CURRENT documents only
    scored = [(len(words_of(state["question"]) & words_of(d["text"])), d["id"]) for d in DOCS if d["status"] == "current"]
    score, best = max(scored)
    return {"context": [best] if score > 1 else [], "trace": ["cid=" + state["cid"] + " retrieve by keywords"]}


def retrieve_by_topic(state):   # second retriever: a topic table. It may find the same document again.
    found = [TOPICS[w] for w in sorted(words_of(state["question"])) if w in TOPICS]
    return {"context": found, "trace": ["cid=" + state["cid"] + " retrieve_by_topic found=" + str(found)]}


def choose_tool(state):
    q = state["question"].lower()
    tool = "freeze_card" if "freeze" in q else "lookup_balance" if "balance" in q else "none"
    return {"tool": tool, "trace": ["cid=" + state["cid"] + " choose_tool tool=" + tool]}


def run_tool(state):   # a local stand-in for the MCP call (built on Days 6-8)
    results = {"lookup_balance": "Balance for cust-1 is 1,250 INR", "freeze_card": "needs human approval", "none": "no tool needed"}
    return {"result": results[state["tool"]], "trace": ["cid=" + state["cid"] + " run_tool " + state["tool"]]}


def write_answer(state):
    return {"answer": "context=" + str(state["context"]) + " result=" + state["result"],
            "trace": ["cid=" + state["cid"] + " write_answer"]}

print("Six workers ready.")

Six workers ready.


## Step 5: BEFORE. The monolith (everyone on one big whiteboard)

All six workers sit in one graph and share one board. A conditional edge (`after_route`) sends policy questions through retrieval first and the rest straight to the tool step.

```text
START -> decide_route --policy?--> retrieve -> retrieve_by_topic -> choose_tool -> run_tool -> write_answer -> END
                      \--other---------------------------------> choose_tool
```

We wrap the build in a function `make_monolith(state_class, topic_node)` so that the experiments can build slightly different versions. For now we call it with the normal ones.

In [6]:
def after_route(state):   # the receptionist's rule: policy questions go through retrieval first
    return "retrieve" if state["route"] == "rag" else "choose_tool"


def make_monolith(state_class=State, topic_node=retrieve_by_topic):
    graph = StateGraph(state_class)
    for name, fn in [("decide_route", decide_route), ("retrieve", retrieve), ("retrieve_by_topic", topic_node),
                     ("choose_tool", choose_tool), ("run_tool", run_tool), ("write_answer", write_answer)]:
        graph.add_node(name, fn)
    graph.add_edge(START, "decide_route")
    graph.add_conditional_edges("decide_route", after_route, {"retrieve": "retrieve", "choose_tool": "choose_tool"})
    graph.add_edge("retrieve", "retrieve_by_topic")
    graph.add_edge("retrieve_by_topic", "choose_tool")
    graph.add_edge("choose_tool", "run_tool")
    graph.add_edge("run_tool", "write_answer")
    graph.add_edge("write_answer", END)
    return graph.compile()


monolith = make_monolith()
print("Monolith built: 6 workers on one board.")

Monolith built: 6 workers on one board.


## Step 6: AFTER. Two departments and a receptionist

### The factory
`make_linear_graph(state_class, nodes)` takes **any list of workers** and chains them into a compiled subgraph, like a template for "a department that does these tasks in order". We use it twice:
- `retrieval_app` = `retrieve` then `retrieve_by_topic` (own board: `RetrievalState`)
- `tool_app` = `choose_tool` then `run_tool` (own board: `ToolState`)

### The wrapper
`call_subgraph` hands the department a **fresh photocopy** (empty reducer boxes) and copies back **only the boxes that department owns**, passing them through the contract guard.

**Why a fresh photocopy?** If you handed over the *old* trace and the department's trace reducer appended to it, the old lines would come back and get added **again**. You will see this bug on purpose in Step 7.

### The receptionist (the parent)
The parent does **only routing**: `decide_route`, then retrieval (if it is a policy question), then the tool path, then `write_answer`.

```text
START -> decide_route --policy?--> retrieval -> tool_path -> write_answer -> END
                      \--other---------------> tool_path
```

In [7]:
def make_linear_graph(state_class, nodes):
    """Reusable factory: any list of workers becomes a compiled subgraph (a chain from START to END)."""
    sub = StateGraph(state_class)
    previous = START
    for node_fn in nodes:
        sub.add_node(node_fn.__name__, node_fn)
        sub.add_edge(previous, node_fn.__name__)
        previous = node_fn.__name__
    sub.add_edge(previous, END)
    return sub.compile()


def call_subgraph(owner, sub_app, sub_input, keys):
    """Reusable wrapper: fresh input in, only the owner's keys out, checked by the contract."""
    out = sub_app.invoke(sub_input)
    return enforce_contract(owner, {key: out[key] for key in keys})


retrieval_app = make_linear_graph(RetrievalState, [retrieve, retrieve_by_topic])
tool_app = make_linear_graph(ToolState, [choose_tool, run_tool])


def run_retrieval(state):
    return call_subgraph("retrieval", retrieval_app,
                         {"question": state["question"], "cid": state["cid"], "context": [], "trace": []},
                         ["context", "trace"])


def run_tool_path(state):
    return call_subgraph("tool_path", tool_app,
                         {"question": state["question"], "cid": state["cid"], "tool": "", "result": "", "trace": []},
                         ["tool", "result", "trace"])


def run_write_answer(state):
    return enforce_contract("write_answer", write_answer(state))


def after_route_parent(state):
    return "retrieval" if state["route"] == "rag" else "tool_path"


def make_parent(state_class=State):
    parent = StateGraph(state_class)
    parent.add_node("decide_route", decide_route)
    parent.add_node("retrieval", run_retrieval)
    parent.add_node("tool_path", run_tool_path)
    parent.add_node("write_answer", run_write_answer)
    parent.add_edge(START, "decide_route")
    parent.add_conditional_edges("decide_route", after_route_parent, {"retrieval": "retrieval", "tool_path": "tool_path"})
    parent.add_edge("retrieval", "tool_path")
    parent.add_edge("tool_path", "write_answer")
    parent.add_edge("write_answer", END)
    return parent.compile()


parent = make_parent()
print("Two departments and a receptionist are ready.")

Two departments and a receptionist are ready.


## Step 7: Prove the refactor changed nothing

**Analogy.** After rearranging an office into departments, you check that customers still get the **same service**. We send the same two questions through both graphs and compare the **entire final whiteboard** with `old == new`.

Then three extra checks:
1. Run the tool department **alone** (a department you can test by itself is the whole point).
2. Add a subgraph **directly** as a node, the shortcut that causes duplicate log lines (the pitfall).
3. Deliberately **break the contract** twice, and watch the guard shout.

**Predict:** for the lost-card question, how many lines will `trace` have? (Count the workers that run: route, two retrievers, choose tool, run tool, write answer.)

In [8]:
def start_state(question, cid):
    return {"user": "cust-1", "team": "support", "question": question, "cid": cid, "t0": 0.0, "route": "",
            "context": [], "tool": "", "result": "", "answer": "", "tokens": 0, "blocked": "", "fallback": False, "trace": []}


for question in ("What happens if my card is lost?", "What is my balance?"):
    old = monolith.invoke(start_state(question, "c1"))
    new = parent.invoke(start_state(question, "c1"))
    print("Question:", question)
    print("  monolith :", old["context"], old["tool"], "|", old["answer"])
    print("  subgraphs:", new["context"], new["tool"], "|", new["answer"])
    print("  trace    :", len(new["trace"]), "lines; refactor keeps behaviour:", old == new)

print("Tool subgraph alone:", tool_app.invoke({"question": "Freeze my card", "cid": "c2", "tool": "", "result": "", "trace": []})["tool"])

Question: What happens if my card is lost?
  monolith : ['P1'] none | context=['P1'] result=no tool needed
  subgraphs: ['P1'] none | context=['P1'] result=no tool needed
  trace    : 6 lines; refactor keeps behaviour: True
Question: What is my balance?
  monolith : [] lookup_balance | context=[] result=Balance for cust-1 is 1,250 INR
  subgraphs: [] lookup_balance | context=[] result=Balance for cust-1 is 1,250 INR
  trace    : 4 lines; refactor keeps behaviour: True
Tool subgraph alone: freeze_card


Look at `context=['P1']`. **Both** retrievers found P1, but `merge_unique` kept it **once**. That is the reducer doing its job.

### The pitfall: adding a subgraph directly

We add `retrieval_app` straight into a graph that shares `trace` with it, and start with one earlier line in the log. The correct answer is **3 lines** (1 earlier + 2 from retrieval).

In [9]:
direct = StateGraph(State)
direct.add_node("retrieval", retrieval_app)     # the shortcut: no fresh photocopy, no wrapper
direct.add_edge(START, "retrieval")
direct.add_edge("retrieval", END)
seed = {**start_state("What happens if my card is lost?", "c3"), "trace": ["cid=c3 earlier line"]}
print("Subgraph added directly, trace lines (3 is correct):", len(direct.compile().invoke(seed)["trace"]))

Subgraph added directly, trace lines (3 is correct): 4


We got **4**: the old line was handed in, the subgraph's reducer appended it again on the way out. The wrapper from Step 6 avoids this.

### The guard shouts

Two bad writes, on purpose: the tool department tries to write `answer` (not its box), and retrieval writes `context` as a plain string (wrong type).

In [10]:
for owner, bad_update in [("tool_path", {"answer": "overwritten!"}), ("retrieval", {"context": "P1"})]:
    try:
        enforce_contract(owner, bad_update)
    except (PermissionError, TypeError) as error:
        print("Contract violation caught:", error)

Contract violation caught: tool_path may not write 'answer'
Contract violation caught: retrieval wrote 'context' as str


### The outputs at a glance

| Line | What it proves | Idea |
| --- | --- | --- |
| `refactor keeps behaviour: True` (twice) | Same final whiteboard after the split | Safe refactor |
| `context=['P1']` | Two retrievers found P1; stored once | Custom reducer |
| `Tool subgraph alone` | A department works without the parent | Testable parts |
| `4` instead of `3` | A shared reducer key got re-merged | Pitfall |
| Two `Contract violation caught` lines | Wrong owner and wrong type both stopped | Ownership contract |

## Step 8: Your turn. Predict first, then run

### Experiment 1: take away the append rule

We build both graphs again, but `trace` becomes a **plain list** (no reducer). **How many trace lines will each graph keep for the lost-card question? And will the equality check still say `True`?**

<details><summary><b>Click to see the answer</b></summary>

**1 line each** (only the last writer's line, `write_answer`), and equality is **still `True`**. Both graphs lost the history in the same way, so they still match. **Equality alone does not prove nothing was lost.** Always look at what is in the box too.
</details>

In [11]:
class StatePlainTrace(State):   # same as State, but trace has NO reducer: overwrite
    trace: list

q = "What happens if my card is lost?"
old = make_monolith(StatePlainTrace).invoke(start_state(q, "e1"))
new = make_parent(StatePlainTrace).invoke(start_state(q, "e1"))
print("monolith trace lines :", len(old["trace"]), old["trace"])
print("subgraph trace lines :", len(new["trace"]))
print("equality still says  :", old == new)

monolith trace lines : 1 ['cid=e1 write_answer']
subgraph trace lines : 1
equality still says  : True


### Experiment 2: append instead of append-if-new

Back to the normal graphs, but we swap the `context` rule from `merge_unique` to plain `operator.add`. **What does `context` hold for the lost-card question?**

<details><summary><b>Click to see the answer</b></summary>

**`['P1', 'P1']`**. Both retrievers find P1, and `operator.add` happily keeps both copies. Choose the reducer from the business rule ("never list a document twice"), not by habit.
</details>

In [12]:
class StateAddContext(State):   # context now uses plain append
    context: Annotated[list, operator.add]

result = make_monolith(StateAddContext).invoke(start_state(q, "e2"))
print("context with operator.add:", result["context"])

context with operator.add: ['P1', 'P1']


### Experiment 3: one retriever finds nothing

`retrieve_by_topic` now returns an empty list. **What happens to `context`?**

<details><summary><b>Click to see the answer</b></summary>

**`['P1']` still.** The first retriever found P1 on its own, and merging in an empty list adds nothing. A merge rule makes the box robust to a worker that finds nothing.
</details>

In [13]:
def retrieve_by_topic_empty(state):
    return {"context": [], "trace": ["cid=" + state["cid"] + " retrieve_by_topic found=[]"]}

result = make_monolith(State, retrieve_by_topic_empty).invoke(start_state(q, "e3"))
print("context when the second retriever finds nothing:", result["context"])

context when the second retriever finds nothing: ['P1']


### Experiment 4: catch your own violation

`write_answer` is allowed to write `answer` and `trace`. It tries to write `tool`. **Which error do you expect, and what does the message say?**

<details><summary><b>Click to see the answer</b></summary>

**`PermissionError: write_answer may not write 'tool'`**. The key is not in its `OWNS` set, so the guard stops it before the value reaches the whiteboard.
</details>

In [14]:
try:
    enforce_contract("write_answer", {"tool": "freeze_card"})
except (PermissionError, TypeError) as error:
    print(type(error).__name__ + ":", error)

PermissionError: write_answer may not write 'tool'


## Step 9: Assignment (refactor a monolithic graph into reusable subgraphs)

Take the single-graph SupportDesk agent (six or more nodes; your L2 Day 20 agent also works). Split it into **two subgraphs plus a parent**, built with a reusable factory like `make_linear_graph`. Deliver:

1. **A table** listing every state key, its type, its owner, its reducer (or "overwrite") and which graphs read it.
2. **A printed run** proving the refactor gives the same result as the monolith.
3. **One standalone run** of each subgraph.
4. **One deliberate contract violation** that is caught.

## Production checklist (you will reuse these lines on Day 19)
- [ ] Every state key has a type, **one named owner** and an explicit **merge rule** in a table.
- [ ] A **run-time contract check** rejects a write to a key the component does not own, and a test proves it.
- [ ] Subgraphs receive **only the input they need** and return **only what they produced**.
- [ ] Each subgraph has a **standalone run**, and a before/after comparison shows results did not change.
- [ ] **Routing** between components lives in the parent graph as visible edges.

## Recap in one minute
| Idea | The analogy | One-line takeaway |
| --- | --- | --- |
| Typed state | A whiteboard with labelled boxes | `TypedDict` is only a hint; check it yourself. |
| Ownership | A sticker saying who may write here | Fail loudly on a wrong writer. |
| Reducer | Erase, add below, or add if new | Pick the rule from the business need. |
| Subgraph | A department with its own board | Small parts you can test alone. |
| Factory | A template for departments | Build many parts with one function. |
| Wrapper | A photocopy in, results out | Avoids re-merging old items. |
| Parent | The receptionist | Routing stays visible in one place. |

## Tomorrow (Day 4)
Today's `trace` lines held only the id and the node. Tomorrow we add **timing and token counts** to them, so you can debug an incident from the trace alone.
